# 14 Post-Selection Probability Study

Study ancilla success probability against maturity, volatility, interest rate, and grid size.


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run 00_project_setup_and_shared_functions.ipynb

config = load_config()
rng = set_global_seed(int(config["random_seed"]))
print("Config loaded and deterministic seed set.")


In [ ]:
spot = 24000.0; K = 24000.0; option_type = "put"
rows = []
for T in [7/365, 14/365, 30/365, 60/365, 120/365]:
    q = quantum_price_reconstruction(spot, K, T, 0.065, 0.18, option_type, n_qubits=7, x_width=float(config["quantum"]["x_width"]))
    rows.append({"study": "maturity", "x": T, "label": f"{T*365:.0f}d", "probability": q["post_selection_probability"]})
for sigma in [0.10, 0.15, 0.20, 0.25, 0.30]:
    q = quantum_price_reconstruction(spot, K, 30/365, 0.065, sigma, option_type, n_qubits=7, x_width=float(config["quantum"]["x_width"]))
    rows.append({"study": "volatility", "x": sigma, "label": f"{sigma:.0%}", "probability": q["post_selection_probability"]})
for r in [0.055, 0.060, 0.065, 0.070, 0.075]:
    q = quantum_price_reconstruction(spot, K, 30/365, r, 0.18, option_type, n_qubits=7, x_width=float(config["quantum"]["x_width"]))
    rows.append({"study": "interest_rate", "x": r, "label": f"{r:.1%}", "probability": q["post_selection_probability"]})
for n in [4, 5, 6, 7, 8, 9]:
    q = quantum_price_reconstruction(spot, K, 30/365, 0.065, 0.18, option_type, n_qubits=n, x_width=float(config["quantum"]["x_width"]))
    rows.append({"study": "grid_size", "x": n, "label": str(n), "probability": q["post_selection_probability"]})
study = pd.DataFrame(rows)
save_table(study, "14_post_selection_probability_study.csv")
for study_name, title, filename in [
    ("maturity", "Post-selection probability versus maturity", "14_post_selection_probability_vs_maturity.png"),
    ("volatility", "Post-selection probability versus volatility", "14_post_selection_probability_vs_volatility.png"),
    ("interest_rate", "Post-selection probability versus interest rate", "14_post_selection_probability_vs_interest_rate.png"),
    ("grid_size", "Post-selection probability versus grid size", "14_post_selection_probability_vs_grid_size.png"),
]:
    group = study[study["study"] == study_name]
    plt.figure()
    plt.plot(group["x"], group["probability"], marker="o")
    plt.title(title)
    plt.xlabel(study_name)
    plt.ylabel("P(ancilla = 0)")
    save_current_figure(filename)
study
